In [6]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import nfl_data_py as nfl

data = nfl.import_pbp_data(range(2025,2026), downcast=True)
data = data[data['play_type'] == 'pass']
passer_df = data.groupby(['passer_player_name', 'game_id', 'passer_player_id', 'week', 'season'], as_index=False).agg(
    {'posteam' : 'first',
     'defteam' : 'first',
     'home_team' : 'first',
     'away_team' : 'first',
     'air_yards' : 'sum',
     'yards_after_catch' : 'sum',
     'epa' : 'sum',
     'complete_pass' : 'sum',
     'incomplete_pass' : 'sum',
     'interception' : 'sum',
     'qb_hit' : 'sum',
     'sack' : 'sum',
     'pass_touchdown' : 'sum',
     'passing_yards' : 'sum',
     'cpoe' : 'mean',
     'roof' : 'first',
     'surface' : 'first'
     }
)

passer_df[(passer_df['season'] == 2025)]
passer_df[['season','week','passer_player_name','passing_yards','pass_touchdown']]


2025 done.
Downcasting floats.


,season,week,passer_player_name,passing_yards,pass_touchdown
0,2025,1,A.Rodgers,244.0,4.0
1,2025,2,A.Rodgers,203.0,1.0
2,2025,1,B.Mayfield,167.0,3.0
3,2025,2,B.Mayfield,215.0,2.0
4,2025,1,B.Nix,176.0,1.0
...,...,...,...,...,...
66,2025,2,T.Lawrence,294.0,3.0
67,2025,1,T.Tagovailoa,114.0,1.0
68,2025,2,T.Tagovailoa,315.0,2.0
69,2025,2,T.Taylor,56.0,1.0


In [7]:
import os
import glob

all_dataframes = []

csv_files = glob.glob(os.path.join(r'C:\Users\rfo7799\Desktop\Git\TetheredAI\NFL\Preds', "*.csv"))
for file_path in csv_files:
    try:
        df = pd.read_csv(file_path)
        all_dataframes.append(df)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
if all_dataframes:
    preds_df = pd.concat(all_dataframes, ignore_index=True)
else:
    print('No csv files to combine')

#preds_df.head()
preds_df[['season','week','passer_player_name','predicted_passing_yards']]


,season,week,passer_player_name,predicted_passing_yards
0,2025,1,J.Hurts,184.282484
1,2025,1,D.Prescott,245.424860
2,2025,1,J.Herbert,290.200261
3,2025,1,P.Mahomes,272.192139
4,2025,1,M.Penix,199.073974
...,...,...,...,...
82,2025,3,K.Murray,244.999112
83,2025,3,R.Wilson,251.896858
84,2025,3,P.Mahomes,257.820263
85,2025,3,L.Jackson,236.540673


In [9]:
combined_df = passer_df[['season','week','passer_player_name','passing_yards']].merge(preds_df[['season','week','passer_player_name','predicted_passing_yards']], on=['season','week','passer_player_name'])
combined_df['passing_yds_var'] = combined_df['predicted_passing_yards'] - combined_df['passing_yards']
from datetime import datetime
combined_df.to_csv(fr"historical_evaluation_{datetime.today().strftime('%Y-%m-%d')}.csv", index=False)